# ※ 과제 안내
- 과제 배점: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.

- 채점 기준:
    - 출력 결과 일치: 제출한 코드가 제시된 출력 결과와 일치하는 경우에만 정답으로 인정됩니다.
    - 코드의 다양성 인정: 출력 결과가 동일하다면 다양한 접근 방식을 존중하여 정답으로 인정합니다.

---
# MCP 실전 과제 — Part 1: 코드 구현 (70점)

Chapter 2~4 핵심 구현 능력을 평가합니다.
각 문제 10점, 총 7문제 70점입니다.

In [ ]:
# 실습 환경 세팅

!pip install -q pyyaml

In [ ]:
# 공통 실행 환경 (모든 문제 공통)

import json
import sqlite3
import hashlib
import re
import time
import asyncio
import inspect
import csv
import io
import os
import tempfile
import typing
import yaml
from pathlib import Path
from dataclasses import dataclass, field, asdict
from enum import Enum
from datetime import datetime, timezone

---
## 문제 1. Tool 기초: 타입 힌트와 JSON Schema [10점]

MCP Tool의 핵심은 Python 함수의 타입 힌트가 자동으로 JSON Schema로 변환되는 것입니다.

`@mcp.tool()` 데코레이터는 내부적으로 함수의 시그니처를 분석하여 Tool 스키마를 생성합니다.
이 원리를 직접 구현해 보세요.

In [ ]:
# 문제 1-1. 타입 힌트 → JSON Schema 변환 함수
#
# Python 타입을 JSON Schema 타입 문자열로 변환하는 함수를 작성하시오.
#   str → "string"
#   int → "integer"
#   float → "number"
#   bool → "boolean"
#   그 외 → "string" (기본값)

# TODO
def python_type_to_json_schema(py_type) -> str:
    pass

In [ ]:
# 문제 1-2. 함수 시그니처에서 MCP Tool 스키마 추출
#
# 주어진 함수에서 다음을 추출하여 딕셔너리로 반환하시오.
#   - name: 함수 이름
#   - description: 함수의 docstring
#   - inputSchema: JSON Schema (properties, required, type)
#
# 조건:
#   - 'ctx' 파라미터는 제외 (MCP Context 주입용)
#   - default 값이 있는 파라미터는 required에서 제외
#   - inspect 모듈을 사용하시오

# TODO
def extract_tool_schema(func) -> dict:
    pass

In [ ]:
# 검증 1

def lookup_inventory(query: str, limit: int = 10, ctx=None) -> str:
    """Search inventory items by keyword."""
    pass

schema = extract_tool_schema(lookup_inventory)
print(json.dumps(schema, indent=2))

# 기대 출력:
# {
#   "name": "lookup_inventory",
#   "description": "Search inventory items by keyword.",
#   "inputSchema": {
#     "type": "object",
#     "properties": {
#       "query": { "type": "string" },
#       "limit": { "type": "integer" }
#     },
#     "required": ["query"]
#   }
# }

---
## 문제 2. 실전 Tool (1): CSV → SQLite 재고 조회 [10점]

MCP 서버에서 재고 조회 Tool은 CSV 파일을 SQLite 인메모리 DB에 로드한 후,
SQL LIKE 검색으로 결과를 반환합니다.

In [ ]:
# 제공 데이터 (수정 금지)

INVENTORY_CSV = """item_id,name,category,quantity,location,status,last_updated
INV-001,MacBook Pro 14 M3 Pro,Electronics,25,Warehouse A,in_stock,2026-01-15
INV-002,Dell Latitude 5540,Electronics,30,Warehouse B,in_stock,2026-01-20
INV-003,Lenovo ThinkPad X1 Carbon,Electronics,15,Warehouse A,in_stock,2026-01-18
INV-004,Dell UltraSharp U2723QE Monitor,Electronics,40,Warehouse C,in_stock,2026-01-10
INV-005,Logitech MX Keys Keyboard,Peripherals,100,Warehouse B,in_stock,2026-01-22
INV-006,Apple Magic Mouse,Peripherals,60,Warehouse A,in_stock,2026-01-12
INV-007,Jabra Evolve2 75 Headset,Peripherals,45,Warehouse C,in_stock,2026-01-25
INV-008,CalDigit TS4 Docking Station,Peripherals,20,Warehouse A,low_stock,2026-01-08
INV-009,iPad Air M2,Electronics,10,Warehouse B,low_stock,2026-01-14
INV-010,HP LaserJet Pro M404dn,Office,8,Warehouse C,in_stock,2026-01-30
INV-011,Office Chair ErgoMax Pro,Furniture,50,Warehouse A,in_stock,2026-02-01
INV-012,Standing Desk VariDesk Pro,Furniture,12,Warehouse B,low_stock,2026-01-28"""

In [ ]:
# 문제 2-1. CSV → SQLite 인메모리 DB 초기화
#
# INVENTORY_CSV 문자열을 파싱하여 SQLite 인메모리 DB에 inventory 테이블을 생성하시오.
#
# 테이블 스키마:
#   item_id TEXT PRIMARY KEY, name TEXT, category TEXT,
#   quantity INTEGER, location TEXT, status TEXT, last_updated TEXT
#
# 조건:
#   - db.row_factory = sqlite3.Row 설정 (딕셔너리처럼 접근)
#   - quantity는 INTEGER로 변환하여 삽입

# TODO
def init_inventory_db(csv_data: str) -> sqlite3.Connection:
    pass

In [ ]:
# 문제 2-2. SQL LIKE 검색 함수
#
# name 또는 category 필드에서 키워드를 검색하는 함수를 구현하시오.
#
# 조건:
#   - 대소문자 무시 (LOWER 사용)
#   - name OR category 매칭
#   - LIMIT 10
#   - 결과를 dict 리스트로 반환 (JSON 직렬화 가능)

# TODO
def search_inventory(db: sqlite3.Connection, query: str) -> list[dict]:
    pass

In [ ]:
# 검증 2

db = init_inventory_db(INVENTORY_CSV)

# 테스트 1: 키워드 검색
results = search_inventory(db, "dell")
print(f"'dell' 검색 결과: {len(results)}건")
for r in results:
    print(f"  - {r['item_id']}: {r['name']} ({r['category']})")

# 테스트 2: 카테고리 검색
results2 = search_inventory(db, "peripherals")
print(f"\n'peripherals' 검색 결과: {len(results2)}건")

# 테스트 3: 결과 없음
results3 = search_inventory(db, "xyz_not_exist")
print(f"\n'xyz_not_exist' 검색 결과: {len(results3)}건")

# 기대 출력:
# 'dell' 검색 결과: 1건
#   - INV-002: Dell Latitude 5540 (Electronics)
#   - INV-004: Dell UltraSharp U2723QE Monitor (Electronics)
#
# 'peripherals' 검색 결과: 4건
#
# 'xyz_not_exist' 검색 결과: 0건

---
## 문제 3. 에러 처리: ToolError 패턴 [10점]

MCP에서는 두 가지 에러가 있습니다:
- **Protocol Error**: JSON-RPC 레벨 에러 (잘못된 메서드, 파싱 실패 등)
- **Tool Error**: 비즈니스 로직 에러 (isError: true)

빈 결과는 에러가 **아닙니다**. 에러와 빈 결과를 정확히 구분해야 합니다.

In [ ]:
# 문제 3-1. ErrorCode Enum과 ToolError 클래스
#
# ErrorCode: NOT_FOUND, INVALID_ARGUMENT, PERMISSION_DENIED, INTERNAL_ERROR, CONFLICT
# ToolError: Exception을 상속하며, code와 message를 받는다.
#   __str__() → "[INVALID_ARGUMENT] 검색어를 입력하세요" 형태

# TODO
class ErrorCode(Enum):
    pass

# TODO
class ToolError(Exception):
    pass

In [ ]:
# 문제 3-2. 안전한 검색 함수
#
# 문제 2에서 만든 db를 사용하여 안전한 검색 함수를 구현하시오.
#
# 규칙:
#   - query가 빈 문자열이거나 공백만 있으면 → ToolError(INVALID_ARGUMENT) raise
#   - 검색 결과가 없으면 → 빈 리스트 반환 (에러가 아님!)
#   - DB 에러 발생 시 → ToolError(INTERNAL_ERROR) raise
#   - 정상 결과 → dict 리스트 반환

# TODO
def safe_search(db: sqlite3.Connection, query: str) -> list[dict]:
    pass

In [ ]:
# 검증 3

# 테스트 1: 정상 검색
results = safe_search(db, "macbook")
print(f"정상 검색: {len(results)}건 - {results[0]['name']}")

# 테스트 2: 빈 결과 (에러 아님)
empty = safe_search(db, "xyz")
print(f"빈 결과: {empty} (type: {type(empty).__name__})")

# 테스트 3: 빈 문자열 → ToolError
try:
    safe_search(db, "")
except ToolError as e:
    print(f"빈 입력 에러: {e}")

# 테스트 4: 공백만 → ToolError
try:
    safe_search(db, "   ")
except ToolError as e:
    print(f"공백 입력 에러: {e}")

# 기대 출력:
# 정상 검색: 1건 - MacBook Pro 14 M3 Pro
# 빈 결과: [] (type: list)
# 빈 입력 에러: [INVALID_ARGUMENT] 검색어를 입력하세요
# 공백 입력 에러: [INVALID_ARGUMENT] 검색어를 입력하세요

---
## 문제 4. 실전 Tool (2): YAML 파싱과 관련도 랭킹 [10점]

정책 검색 Tool은 Markdown 문서에서 YAML front-matter를 파싱하고,
가중치 기반으로 관련도 점수를 계산하여 랭킹합니다.

In [ ]:
# 제공 데이터 (수정 금지)

POLICY_DOC_1 = """---
title: VPN 설정 가이드
tags: [vpn, 네트워크, 보안, 원격근무]
last_updated: 2026-01-15
---
# VPN 설정 가이드

## 개요
사내 VPN은 WireGuard를 기본으로 사용합니다.

## 설치 방법
macOS에서는 Homebrew로 설치합니다.
VPN 연결이 필요한 경우 IT팀에 문의하세요.
VPN 설정 후 반드시 연결 테스트를 수행하세요."""

POLICY_DOC_2 = """---
title: 보안 정책
tags: [보안, 비밀번호, 인증, 컴플라이언스]
last_updated: 2026-02-01
---
# 보안 정책

## 비밀번호 규칙
비밀번호는 14자 이상이어야 합니다.

## MFA 필수
모든 직원은 MFA를 활성화해야 합니다.
VPN 접속 시에도 MFA 인증이 필요합니다."""

POLICY_DOC_3 = """---
title: 원격근무 정책
tags: [원격근무, 재택, 하이브리드]
last_updated: 2026-01-20
---
# 원격근무 정책

## 핵심 근무 시간
10:00~16:00 사이에는 반드시 연락 가능해야 합니다.

## 장비 지원
노트북, 모니터 1대, 키보드, 마우스, 헤드셋을 지원합니다."""

In [ ]:
# 문제 4-1. YAML Front-matter 파싱
#
# Markdown 문서에서 YAML front-matter와 본문을 분리하는 함수를 구현하시오.
#
# 입력: "---\ntitle: ...\ntags: [...]\n---\n본문..."
# 출력: (meta_dict, body_str)
#   - meta_dict: YAML 파싱 결과 딕셔너리
#   - body_str: front-matter 이후의 본문 문자열
#
# front-matter가 없으면 ({}, 전체 문자열) 반환

# TODO
def parse_frontmatter(content: str) -> tuple[dict, str]:
    pass

In [ ]:
# 문제 4-2. 관련도 점수 계산
#
# 검색 키워드에 대해 문서의 관련도 점수를 계산하는 함수를 구현하시오.
#
# 점수 규칙 (대소문자 무시):
#   - 제목(title)에 키워드 포함: +3.0점
#   - 태그(tags) 중 하나라도 키워드 포함: +2.0점
#   - 본문(body)에서 키워드 등장 횟수 × 0.5 (최대 3.0점)

# TODO
def calculate_relevance(query: str, title: str, tags: list[str], body: str) -> float:
    pass

In [ ]:
# 검증 4

# 파싱 테스트
meta1, body1 = parse_frontmatter(POLICY_DOC_1)
print(f"제목: {meta1['title']}")
print(f"태그: {meta1['tags']}")
print(f"본문 길이: {len(body1)}자")

# 관련도 테스트: "vpn"으로 검색
docs = [POLICY_DOC_1, POLICY_DOC_2, POLICY_DOC_3]
print("\n=== 'vpn' 검색 관련도 ===")
for i, doc in enumerate(docs, 1):
    meta, body = parse_frontmatter(doc)
    score = calculate_relevance("vpn", meta["title"], meta["tags"], body)
    print(f"문서{i} ({meta['title']}): {score}점")

# 기대 출력:
# 제목: VPN 설정 가이드
# 태그: ['vpn', '네트워크', '보안', '원격근무']
# 본문 길이: (약 120~140자)
#
# === 'vpn' 검색 관련도 ===
# 문서1 (VPN 설정 가이드): 7.0점  (제목3 + 태그2 + 본문4×0.5=2.0)
# 문서2 (보안 정책): 0.5점         (본문에 VPN 1회)
# 문서3 (원격근무 정책): 0.0점

---
## 문제 5. 실전 Tool (3): Confirm Gate와 멱등성 [10점]

상태를 변경하는 Tool(티켓 생성 등)은 반드시 안전 장치가 필요합니다.
- **Confirm Gate**: preview → confirm 2단계로 실수 방지
- **멱등성 키**: 동일 요청의 중복 실행 방지

In [ ]:
# 문제 5-1. Confirm Gate 패턴 구현
#
# 티켓 생성 함수를 구현하시오.
#
# confirm=False일 때:
#   - 데이터를 변경하지 않고 미리보기 문자열만 반환
#   - 반환: "[미리보기] 제목: {title} | 우선순위: {priority} — confirm=True로 생성하세요."
#
# confirm=True일 때:
#   - tickets_store 리스트에 티켓 딕셔너리 추가
#   - ticket_id: "TKT-001", "TKT-002" ... 형식 (tickets_store 길이 + 1 기반)
#   - 반환: 생성된 티켓 딕셔너리
#     {"ticket_id", "title", "body", "priority", "status": "open",
#      "created_at": ISO format UTC}

# TODO
def create_ticket(
    title: str,
    body: str,
    priority: str,
    confirm: bool,
    tickets_store: list
) -> str | dict:
    pass

In [ ]:
# 문제 5-2. 멱등성 키 로직 추가
#
# 위 함수를 확장하여 멱등성을 보장하는 함수를 구현하시오.
#
# 멱등성 키: hashlib.sha256((title + body).encode()).hexdigest()[:16]
#
# confirm=True일 때:
#   1. 멱등성 키 생성
#   2. tickets_store에서 동일 키를 가진 티켓이 있는지 확인
#   3. 있으면 → 기존 티켓 반환 (중복 생성 방지)
#   4. 없으면 → 새 티켓 생성 (idempotency_key 필드 포함)

# TODO
def create_ticket_idempotent(
    title: str,
    body: str,
    priority: str,
    confirm: bool,
    tickets_store: list
) -> str | dict:
    pass

In [ ]:
# 검증 5

store = []

# 테스트 1: 미리보기
preview = create_ticket_idempotent("프린터 고장", "2층 프린터가 작동하지 않습니다.", "medium", confirm=False, tickets_store=store)
print(f"미리보기: {preview}")
print(f"저장소 변경 없음: {len(store)}건")

# 테스트 2: 생성
ticket1 = create_ticket_idempotent("프린터 고장", "2층 프린터가 작동하지 않습니다.", "medium", confirm=True, tickets_store=store)
print(f"\n생성된 티켓: {ticket1['ticket_id']} - {ticket1['title']}")
print(f"저장소: {len(store)}건")

# 테스트 3: 중복 생성 시도 (동일 title+body)
ticket2 = create_ticket_idempotent("프린터 고장", "2층 프린터가 작동하지 않습니다.", "medium", confirm=True, tickets_store=store)
print(f"\n중복 시도 결과: {ticket2['ticket_id']} (기존 티켓 반환)")
print(f"저장소 변경 없음: {len(store)}건")

# 테스트 4: 다른 티켓 생성
ticket3 = create_ticket_idempotent("모니터 깜빡임", "3층 모니터가 깜빡입니다.", "high", confirm=True, tickets_store=store)
print(f"\n새 티켓: {ticket3['ticket_id']} - {ticket3['title']}")
print(f"저장소: {len(store)}건")

# 기대 출력:
# 미리보기: [미리보기] 제목: 프린터 고장 | 우선순위: medium — confirm=True로 생성하세요.
# 저장소 변경 없음: 0건
#
# 생성된 티켓: TKT-001 - 프린터 고장
# 저장소: 1건
#
# 중복 시도 결과: TKT-001 (기존 티켓 반환)
# 저장소 변경 없음: 1건
#
# 새 티켓: TKT-002 - 모니터 깜빡임
# 저장소: 2건

---
## 문제 6. 입력 검증과 보안 [10점]

LLM이 생성하는 입력은 신뢰할 수 없습니다.
Prompt Injection을 통해 악의적인 입력이 Tool에 전달될 수 있으므로,
모든 입력을 검증해야 합니다.

In [ ]:
# 제공 상수 (수정 금지)

CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

DANGEROUS_PATTERNS = [
    re.compile(r"['\\";]\s*(?:DROP|DELETE|UPDATE|INSERT|ALTER|EXEC)", re.IGNORECASE),
    re.compile(r"<\s*script", re.IGNORECASE),
    re.compile(r"\$(?:gt|lt|ne|eq|regex|where|or|and)\b"),
    re.compile(r"__proto__|constructor\s*\["),
]

In [ ]:
# 문제 6-1. sanitize_string() 구현
#
# 입력 문자열을 정제하는 함수를 구현하시오.
#
# 처리 순서:
#   1. 타입 검사: str이 아니면 ValueError
#   2. 앞뒤 공백 제거 (strip)
#   3. 제어 문자 제거 (CONTROL_CHARS 사용)
#   4. max_length 초과 시 잘라내기
#   5. DANGEROUS_PATTERNS 중 하나라도 매칭되면 ValueError
#   6. 정제된 문자열 반환

# TODO
def sanitize_string(value: str, max_length: int = 500) -> str:
    pass

In [ ]:
# 문제 6-2. validate_doc_id() 구현
#
# MCP Resource URI의 doc_id를 검증하는 함수를 구현하시오.
#
# 화이트리스트 패턴: ^[a-z0-9]([a-z0-9\-]{0,48}[a-z0-9])?$
#   - 소문자 영숫자와 하이픈만 허용, 1~50자
#   - 하이픈으로 시작/끝 불가
#
# 차단: .., /, \, 널 바이트, 대문자, 빈 문자열
# 통과 시 strip된 doc_id 반환, 실패 시 ValueError

# TODO
def validate_doc_id(doc_id: str) -> str:
    pass

In [ ]:
# 검증 6

print("=== sanitize_string ===")
print(f"정상: '{sanitize_string('  hello world  ')}'")
print(f"길이제한: '{sanitize_string('a' * 600, max_length=10)}'")

for attack in ["'; DROP TABLE users; --", "<script>alert(1)</script>", "$gt value"]:
    try:
        sanitize_string(attack)
        print(f"통과 (오류!): {attack}")
    except ValueError:
        print(f"차단됨: {attack[:30]}")

print("\n=== validate_doc_id ===")
for valid_id in ["vpn-setup", "remote-work", "a", "abc123"]:
    print(f"통과: {validate_doc_id(valid_id)}")

for bad_id in ["../../../etc/passwd", "foo/bar", "UPPER", "", "a" * 60, "hello\x00world"]:
    try:
        validate_doc_id(bad_id)
        print(f"통과 (오류!): {bad_id}")
    except ValueError:
        print(f"차단됨: {repr(bad_id)[:30]}")

# 기대 출력:
# === sanitize_string ===
# 정상: 'hello world'
# 길이제한: 'aaaaaaaaaa'
# 차단됨: '; DROP TABLE users; --
# 차단됨: <script>alert(1)</script>
# 차단됨: $gt value
# === validate_doc_id ===
# 통과: vpn-setup / remote-work / a / abc123
# 차단됨: (각 공격 패턴)

---
## 문제 7. Resource 구현 + Prompt 템플릿 [10점]

MCP Resource는 읽기 전용 데이터를 URI로 노출하고,
Prompt는 재사용 가능한 지시문 템플릿입니다.
정적 리소스와 동적 리소스를 구현하고, 프롬프트 템플릿을 작성합니다.

In [ ]:
# 제공 데이터 구조 (수정 금지)

@dataclass
class PolicyDoc:
    doc_id: str
    title: str
    path: Path
    tags: list[str]

In [ ]:
# 문제 7-1. 정책 인덱스 + 상세 리소스
#
# build_policy_index(policies): PolicyDoc 리스트 → [{"doc_id", "title", "tags"}] (path 제외)
#
# get_policy_content(policies, doc_id, policy_dir):
#   1. validate_doc_id()로 보안 검증 (문제 6 함수 사용)
#   2. doc_id로 PolicyDoc 찾기, 없으면 ValueError
#   3. 파일 읽기 → {"doc_id", "title", "content"} 반환

# TODO
def build_policy_index(policies: list[PolicyDoc]) -> list[dict]:
    pass

# TODO
def get_policy_content(policies: list[PolicyDoc], doc_id: str, policy_dir: Path) -> dict:
    pass

In [ ]:
# 문제 7-2. Prompt 템플릿
#
# IT 인시던트 보고서 프롬프트를 생성하는 함수를 구현하시오.
#
# 반환할 프롬프트에 반드시 포함:
#   "요약", "영향 범위", "재현 절차", "권장 조치", "우선순위"
# issue와 affected_system을 본문에 포함

# TODO
def incident_report_prompt(issue: str, affected_system: str) -> str:
    pass

In [ ]:
# 검증 7

# Resource 테스트
tmp = Path(tempfile.mkdtemp())
(tmp / "vpn-setup.md").write_text(POLICY_DOC_1, encoding="utf-8")
(tmp / "security-policy.md").write_text(POLICY_DOC_2, encoding="utf-8")

policies = [
    PolicyDoc("vpn-setup", "VPN 설정 가이드", tmp / "vpn-setup.md", ["vpn", "네트워크"]),
    PolicyDoc("security-policy", "보안 정책", tmp / "security-policy.md", ["보안", "인증"]),
]

index = build_policy_index(policies)
print(f"인덱스: {len(index)}개")
print(json.dumps(index, ensure_ascii=False, indent=2))

detail = get_policy_content(policies, "vpn-setup", tmp)
print(f"\n상세: {detail['doc_id']} - {detail['title']} ({len(detail['content'])}자)")

try:
    get_policy_content(policies, "../../etc/passwd", tmp)
except ValueError:
    print("Path Traversal 차단됨")

# Prompt 테스트
prompt = incident_report_prompt("2층 프린터 작동 중단", "HP LaserJet Pro M404dn")
for item in ["요약", "영향 범위", "재현 절차", "권장 조치", "우선순위"]:
    print(f"  {item}: {'✓' if item in prompt else '✗'}")
print(f"  이슈 포함: {'✓' if '프린터' in prompt else '✗'}")
print(f"  시스템 포함: {'✓' if 'LaserJet' in prompt else '✗'}")

# 기대 출력:
# 인덱스: 2개
# [{"doc_id": "vpn-setup", ...}, {"doc_id": "security-policy", ...}]
# 상세: vpn-setup - VPN 설정 가이드
# Path Traversal 차단됨
# 요약: ✓ / 영향 범위: ✓ / ... 모두 ✓

# MCP 실전 과제 — Part 2: 킬러 문항 (30점)

> **범위**: Chapter 2~4 (Tools, Resources, Prompts, Testing, Client Integration)

---

## 시험 구조

총 3문제 (30점 만점) | 문제당 10점
- 각 문제는 X-1, X-2 두 개의 세부문제로 구성됩니다.
- **X-1, X-2 중 하나라도 틀리면 해당 문제 전체 0점**

## 증빙 원칙

- 모든 캡처 시 반드시 **`date && hostname`** 명령 출력 포함
- 중간 결과(로그, 캡처, diff) 제출 필수 — 최종 결과만 제출 시 0점

캡처 예시 (date && hostname 포함):
```
$ date && hostname
Thu Apr 17 19:30:00 KST 2026
ubuntu-de-lab
```

## 과제 제출 안내

> **아래 경우 모두 0점 처리**
> - .docx / .hwp 등 문서 파일만 단독 제출
> - 코드를 이미지(캡처/사진)로 제출

반드시 아래 폴더 구조를 지켜 ZIP 파일로 압축하여 제출하세요.
- 각 문제 폴더(problem-8/ ~ problem-10/) 전체를 포함하여 ZIP 제출
- 코드, 설정 파일, 캡처, 텍스트 증빙을 모두 폴더 안에 포함할 것

파일명: MCP_몇기_이름_과제.zip  
예시: MCP_1기_홍길동_과제.zip

---

## 문제 8: MCP 서버 구현 + Inspector 검증 [10점]

주제: 서버 구동 + 외부 도구 연동 | 난이도: 중급

### 상황

> starter-kit의 TODO를 완성하여 MCP 서버를 구동하고,
> MCP Inspector로 Tool과 Resource가 정상 동작하는지 검증합니다.
>
> - starter-kit 기반으로 시작
> - `lookup_inventory` Tool과 `policy://index` Resource 최소 구현
> - MCP Inspector로 연결하여 실제 호출 테스트

### 8-1: MCP 서버 구현 및 Inspector 연결

starter-kit에서 `lookup_inventory` Tool과 `policy://index` Resource를 구현하고,
MCP Inspector로 연결하여 등록된 Tool/Resource 목록을 확인하시오.

#### 요구사항
- starter-kit의 TODO를 완성하여 서버 구동
- `npx @modelcontextprotocol/inspector`로 서버에 연결
- Inspector에서 Tool 목록과 Resource 목록이 보이는 화면 캡처

#### 평가 기준

| # | 통과 조건 (코드/설명) | 통과 조건 (캡처/증빙) |
|---|---|---|
| 8-1 | 완성된 server.py + tool/resource 코드 제출 | Inspector에서 Tool/Resource 목록 캡처 |

### 8-2: Inspector에서 Tool 호출 테스트

Inspector에서 `lookup_inventory` Tool을 직접 호출하고 JSON-RPC 메시지를 확인하시오.

#### 요구사항
- Inspector에서 `lookup_inventory`를 `{"query": "macbook"}` 파라미터로 호출
- JSON-RPC Request와 Response가 보이는 화면 캡처
- 결과에 MacBook Pro 아이템이 포함되어야 함

#### 평가 기준

| # | 통과 조건 (코드/설명) | 통과 조건 (캡처/증빙) |
|---|---|---|
| 8-2 | Tool 호출 파라미터 + 결과 JSON 제출 | Inspector Tool 호출 결과 캡처 + JSON-RPC 메시지 캡처 |

### 제출물

```
problem-8/answer/
├── src/                        # 완성된 서버 코드
├── inspector_tools.png         # Tool 목록 캡처 (date && hostname)
├── inspector_resources.png     # Resource 목록 캡처
├── inspector_tool_call.png     # Tool 호출 결과 캡처
└── jsonrpc_messages.txt        # Request/Response JSON 텍스트
```

> **0점 조건 (세부문제 중 하나라도 미충족 시 해당 문제 전체 0점)**
> - MCP Inspector 캡처 없음
> - Tool 호출 결과에 MacBook 아이템 미포함
> - 캡처에 date && hostname 출력 미포함
> - 서버 코드 미제출

---

## 문제 9: Streamable HTTP Transport + curl 검증 [10점]

주제: Transport 전환 + 원격 접속 검증 | 난이도: 심화

### 상황

> stdio Transport로 동작하는 MCP 서버를 Streamable HTTP로 전환하고,
> curl로 직접 JSON-RPC 요청을 보내 동작을 검증합니다.
>
> - stdio → Streamable HTTP Transport 전환
> - 포트: 8000
> - curl로 `tools/list`, `tools/call` 요청

### 9-1: Streamable HTTP Transport 전환

서버를 Streamable HTTP Transport로 전환하고 실행하시오.

#### 요구사항
- `server.py`에서 Transport 전환 코드 작성
- `uv run python src/server.py --transport streamable-http --port 8000` 실행
- 서버 시작 로그 캡처 (포트 바인딩 확인)

#### 평가 기준

| # | 통과 조건 (코드/설명) | 통과 조건 (캡처/증빙) |
|---|---|---|
| 9-1 | Transport 전환 코드 제출 | 서버 시작 로그 캡처 (포트 8000 확인) |

### 9-2: curl로 JSON-RPC 요청 테스트

curl로 MCP 서버에 JSON-RPC 요청을 보내 Tool 목록 조회 및 Tool 호출을 수행하시오.

#### 요구사항

**요청 1: Tool 목록 조회**
```bash
curl -X POST http://localhost:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","id":1,"method":"tools/list","params":{}}'
```

**요청 2: Tool 호출**
```bash
curl -X POST http://localhost:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"lookup_inventory","arguments":{"query":"keyboard"}}}'
```

- 각 요청의 Response 전문 캡처
- `tools/list` 응답에 `lookup_inventory` 포함 확인
- `tools/call` 응답에 검색 결과(Keyboard 아이템) 포함 확인

#### 평가 기준

| # | 통과 조건 (코드/설명) | 통과 조건 (캡처/증빙) |
|---|---|---|
| 9-2a | curl 명령어 제출 | tools/list 응답 캡처 |
| 9-2b | 응답 JSON 분석 | tools/call 결과에 Keyboard 아이템 포함 캡처 |

### 제출물

```
problem-9/answer/
├── server.py                   # Transport 전환 코드
├── server_start.png            # 서버 시작 로그 캡처 (date && hostname)
├── curl_tools_list.txt         # tools/list 요청+응답
├── curl_tool_call.txt          # tools/call 요청+응답
└── curl_results.png            # curl 실행 결과 캡처
```

> **0점 조건 (세부문제 중 하나라도 미충족 시 해당 문제 전체 0점)**
> - Streamable HTTP 미사용 (stdio로만 실행)
> - curl 요청/응답 증빙 없음
> - tools/list 응답에 lookup_inventory 미포함
> - 캡처에 date && hostname 출력 미포함

---

## 문제 10: pytest 전체 테스트 + 커버리지 90% 이상 [10점]

주제: 테스팅 + 정량 수치 검증 | 난이도: 중급

### 상황

> 구현한 MCP 서버의 품질을 보장하기 위해 테스트 코드를 작성하고,
> 코드 커버리지 90% 이상을 달성해야 합니다.
>
> - pytest + pytest-cov 사용
> - 최소 테스트 11개 이상
> - 커버리지 90% 이상 필수

### 10-1: 테스트 코드 작성 및 전체 PASSED

다음 최소 요구 테스트를 작성하고 전체 PASSED를 달성하시오.

#### 요구사항

| 테스트 파일 | 최소 테스트 수 | 필수 테스트 |
|---|---|---|
| `test_inventory.py` | 3개 | 정상 검색, 빈 결과, 빈 쿼리 에러 |
| `test_validation.py` | 5개 | SQL injection 차단, XSS 차단, Path Traversal 차단, 정상 입력 통과, doc_id 검증 |
| `test_ticket.py` | 3개 | preview 동작, create 동작, 멱등성 검증 |

- 총 테스트 **11개 이상**
- 전체 **PASSED** (FAILED 0건)

#### 평가 기준

| # | 통과 조건 (코드/설명) | 통과 조건 (캡처/증빙) |
|---|---|---|
| 10-1 | 테스트 코드 11개+ 제출 | pytest 실행 결과 전체 PASSED 캡처 |

### 10-2: 커버리지 90% 이상 달성

`pytest --cov=src --cov-report=term-missing -v` 실행하여 커버리지 90% 이상을 달성하시오.

#### 요구사항
- `uv run pytest --cov=src --cov-report=term-missing -v` 실행
- 전체 커버리지 **90% 이상** 달성

#### 평가 기준

| # | 통과 조건 (코드/설명) | 통과 조건 (캡처/증빙) |
|---|---|---|
| 10-2 | — (자동채점) | 커버리지 90%+ 리포트 캡처 (수치 확인) |

### 제출물

```
problem-10/answer/
├── tests/
│   ├── test_inventory.py
│   ├── test_validation.py
│   └── test_ticket.py
├── pytest_results.png          # 전체 PASSED 캡처 (date && hostname)
├── coverage_report.png         # 커버리지 90%+ 캡처
└── pytest_output.txt           # 실행 로그 전문
```

> **0점 조건 (세부문제 중 하나라도 미충족 시 해당 문제 전체 0점)**
> - 테스트 10개 이하
> - FAILED 테스트 존재
> - 커버리지 90% 미만
> - 캡처에 date && hostname 출력 미포함
